In [ ]:

# Test the live path on Colab. No C++ changed on the colab branch, so the three
# library files are overlaid onto the installed 3.1.10 wheel rather than
# rebuilding the extension (which takes ten minutes).
import os, subprocess, sys, time
RAW = 'https://raw.githubusercontent.com/sokrypton/alphafold3/colab'
import importlib.metadata as md
root = os.path.dirname(md.distribution('alphafold3-colabfold').locate_file('alphafold3'))
for rel in ('src/alphafold3/model/model.py',
            'src/alphafold3/model/network/diffusion_head.py',
            'src/alphafold3/model/atom_layout/atom_layout.py'):
    dst = os.path.join(root, rel.split('src/', 1)[1])
    assert os.system(f'wget -q -O {dst} {RAW}/{rel}') == 0, rel
assert os.system(f'wget -q -O run_alphafold.py {RAW}/run_alphafold.py') == 0
assert os.system(f'wget -q -O live_frames.py {RAW}/dev/live/live_frames.py') == 0
print('overlaid the colab branch onto the installed wheel', flush=True)

import numpy as np, jax
from absl import flags
sys.path.insert(0, '.')
import run_alphafold as RA
flags.FLAGS(['run_alphafold.py', '--norun_data_pipeline',
             '--flash_attention_implementation=xla', '--cache_dir=/tmp/af3_cache'])
import live_frames as LF
from alphafold3.common import folding_input
from alphafold3.constants import decoded_ccd
from alphafold3.data import featurisation
from alphafold3.model import model_registry, weights as _w
from alphafold3.model.components import utils as _u
from alphafold3.model.pipeline import model_features

seq = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK'
import json as _j
_j.dump({'dialect': 'alphafold3', 'version': 4, 'name': 'live', 'modelSeeds': [1],
         'sequences': [{'protein': {'id': 'A', 'sequence': seq,
                                    'unpairedMsa': f'>q\n{seq}\n',
                                    'pairedMsa': '', 'templates': []}}]},
        open('live.json', 'w'))
fi = folding_input.load_fold_inputs_from_path('live.json').__next__()
cfg = RA.make_model_config(model_name='openbind0', num_recycles=3,
                           num_diffusion_samples=1,
                           flash_attention_implementation='xla')
cfg.heads.diffusion.eval.steps = 8
cfg.heads.diffusion.eval.stepwise = True
runner = RA.ModelRunner(config=cfg, device=jax.local_devices()[0],
                        model_dir=_w.default_dir('openbind0', 'int8'))
ccd = decoded_ccd.get_ccd()
feat = lambda: featurisation.featurise_input(fold_input=fi, ccd=ccd,
                                             buckets=None, verbose=False)
raw = feat()[0]
spec = model_registry.get('openbind0')
if spec.featurise:
    raw = model_features.apply(raw, spec, refeaturise=lambda: feat()[0],
                               model_dir=_w.default_dir('openbind0', 'int8'),
                               esm=None, has_msa=True, fold_input=fi, cyclic=False)
bobj = LF.as_batch(raw)
batch = jax.device_put(jax.tree.map(jax.numpy.asarray,
                                    _u.remove_invalidly_typed_feats(raw)))

# Does py2Dmol accept a streamed frame at all, headless?
import py2Dmol
viewer = py2Dmol.view(size=(400, 400), style='cartoon')
viewer.show()
n_rec, n_dif, errs, t0 = [0], [0], [], time.time()
cm_last = [None]
def on_frame(kind, i, data):
    if kind == 'recycle':
        cm_last[0] = LF.contact_map(data['contacts'])
        n_rec[0] += 1
        print(f'  [{time.time()-t0:5.1f}s] recycle {i} contact {cm_last[0].shape}', flush=True)
    else:
        xyz, ch, rid = LF.frame_positions(np.asarray(data)[0], bobj)
        try:
            viewer.add(xyz, chains=ch, residue_numbers=rid,
                       maps={'contact': cm_last[0]})
            n_dif[0] += 1
        except Exception as e:
            errs.append(f'{type(e).__name__}: {e}')
        if i % 3 == 0:
            print(f'  [{time.time()-t0:5.1f}s] diffusion {i} -> py2Dmol '
                  f'({len(xyz)} positions)', flush=True)

res = runner.live_model()(jax.random.PRNGKey(1), batch, on_frame=on_frame)
print(f'RESULT: {n_rec[0]} recycle frames, {n_dif[0]} frames into py2Dmol, '
      f'{len(errs)} viewer errors, {time.time()-t0:.0f}s')
for e in errs[:3]:
    print('   viewer error:', e)
print('PASS' if (n_rec[0] == 4 and n_dif[0] == 8 and not errs) else 'FAIL')
